In [ ]:
import datetime
import pandas as pd
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
from os import listdir
from scipy.interpolate import RegularGridInterpolator
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, Executor
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from tqdm import tqdm_notebook

In [ ]:
## 从NC文件时间转PD时间
def cftime2pdtime(cf):
    return pd.to_datetime(datetime.datetime(cf.year,cf.month,cf.day,cf.hour))

## 获取文件夹里所有NC文件
def getfiles(filed):
    files=listdir(filed)
    files.sort()
    files=files[0:]
    files_n=[filed+'/'+i for i in files]
    return files_n

## 计算等间距6H日平均
def getDaymen(test,times):
    data = test
    time_index = times

    # 将时间序列转化为日期序列，使用 groupby 方法对每天的数据进行分组，最后求平均值
    date_index = time_index.date
    grouped = pd.DataFrame(data.reshape(-1, test.shape[1]*test.shape[2])).groupby(date_index).mean()
    result = grouped.values.reshape(-1, test.shape[1],test.shape[2])
    return result

das1=nc.Dataset('/lustre/home/yuhanxue/data/BRAN/ERA5_4Q/1993_150E_180.nc')
das2=nc.Dataset('/lustre/home/yuhanxue/data/BRAN/ERA5_4Q/1993_0_70W.nc')
lon1=np.array(das1['longitude'])
lon2=np.array(das2['longitude'])
lat=np.array(das1['latitude'])
lon=np.concatenate([lon1[:-1],lon2+360])
print(das1.variables.keys())

In [ ]:
pd.to_datetime(list(map(cftime2pdtime,nc.num2date(das1['time'],das1['time'].units))))

In [ ]:
times=[]
for i  in tqdm_notebook(range(1993,2023)):
    das1=nc.Dataset(f"/lustre/home/yuhanxue/data/BRAN/ERA5_4Q/{i:04d}_150E_180.nc")
    times.append(pd.to_datetime(list(map(cftime2pdtime,nc.num2date(das1['time'],das1['time'].units)))))
times=pd.to_datetime(np.concatenate(times, axis=0))

In [12]:
lon_e=np.arange(np.min(lon),np.max(lon)+1,2)
lat_e=np.arange(np.min(lat), np.max(lat)+1, 2)
lon_4=lon_e
lat_4=lat_e
lon_12=lon
lat_12=lat
Lon_4,Lat_4=np.meshgrid(lon_4,lat_4)
print(f'lat_4 range:{lat_4[0]:.2f}~{lat_4[-1]:.2f} | freq:{abs(lat_4[1]-lat_4[0])}\nlon_4 range:{lon_4[0]:.2f}~{lon_4[-1]:.2f} | freq:{lon_4[1]-lon_4[0]}')
print(f'lat_12 range:{lat_12[0]:.2f}~{lat_12[-1]:.2f} | freq:{abs(lat_12[1]-lat_12[0])}\nlon_12 range:{lon_12[0]:.2f}~{lon_12[-1]:.2f} | freq:{lon_12[1]-lon_12[0]}')
def grid(dat):
    global lon_12,lat_12,Lon_4,Lat_4
    #rint(dat.shape)
    #f=RegularGridInterpolator((lat_12,lon_12),godas_mlds[1,:,:],bounds_error=False,method='qubic',fill_value=np.nan)
    return RegularGridInterpolator((lat_12,lon_12),dat,bounds_error=False,fill_value=np.nan)((Lat_4,Lon_4))
def list_map(dat):
    pool = ThreadPoolExecutor(max_workers=3)
    ans=np.array(list(pool.map(grid,dat)))
    #print(ans.shape)
    del pool
    return ans
# i=1995
# das1=nc.Dataset(f"/lustre/home/yuhanxue/data/BRAN/ERA5_4Q/{i:04d}_150E_180.nc")
# das2=nc.Dataset(f"/lustre/home/yuhanxue/data/BRAN/ERA5_4Q/{i:04d}_0_70W.nc")
# list_map(np.concatenate([das1['mslhf'],das2['mslhf'][:,:,1:]],axis=-1)).shape

lat_4 range:20.00~60.00 | freq:2.0
lon_4 range:150.00~250.00 | freq:2.0
lat_12 range:60.00~20.00 | freq:0.25
lon_12 range:150.00~250.00 | freq:0.25


In [13]:
mslhfs=[]
msnlwrfs=[]
msnswrfs=[]
msshfs=[]
for i  in tqdm_notebook(range(1993,2023)):
    das1=nc.Dataset(f"/lustre/home/yuhanxue/data/BRAN/ERA5_4Q/{i:04d}_150E_180.nc")
    das2=nc.Dataset(f"/lustre/home/yuhanxue/data/BRAN/ERA5_4Q/{i:04d}_0_70W.nc")
    if np.abs(np.sum(das1['mslhf'][:,:,-1]-das2['mslhf'][:,:,0]))>1000:
        print(i)
    mslhf=list_map(np.concatenate([das1['mslhf'],das2['mslhf'][:,:,1:]],axis=-1))
    mslhfs.append(mslhf)
    msnlwrf=list_map(np.concatenate([das1['msnlwrf'],das2['msnlwrf'][:,:,1:]],axis=-1))
    msnlwrfs.append(msnlwrf)
    msnswrf=list_map(np.concatenate([das1['msnswrf'],das2['msnswrf'][:,:,1:]],axis=-1))
    msnswrfs.append(msnswrf)
    msshf=list_map(np.concatenate([das1['msshf'],das2['msshf'][:,:,1:]],axis=-1))
    msshfs.append(msshf)
    
    

/tmp/ipykernel_413107/3542042728.py:5: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for i  in tqdm_notebook(range(1993,2023)):


  0%|          | 0/30 [00:00<?, ?it/s]

In [14]:
LH=getDaymen(np.array(np.concatenate(mslhfs, axis=0)),times)
SH=getDaymen(np.array(np.concatenate(msshfs, axis=0)),times)
LW=getDaymen(np.array(np.concatenate(msnlwrfs, axis=0)),times)
SW=getDaymen(np.array(np.concatenate(msnswrfs, axis=0)),times)

In [16]:
np.savez('ERAdata.npz',LH=LH,SH=SH,LW=LW,SW=SW,lon=lon_e,lat=lat_e)

In [ ]:
das1=nc.Dataset(rf"D:\OneDrive\heat_budget\U&V&SLP150_180.nc")
das2=nc.Dataset(rf"D:\OneDrive\heat_budget\U&V&SLP180_250.nc")
lon1=np.array(das1['longitude'])
lon2=np.array(das2['longitude'])
lat=np.array(das1['latitude'])
lon=np.concatenate([lon1[:-1],lon2+360])
lon

In [ ]:
das1.variables.keys()

In [ ]:
u101=np.arra